In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

model = xgb.Booster()
model.load_model('loan_repayment_model_day1.json') 
test = pd.read_csv('data/test.csv')
test_ids = test['id']

def prepare_submission_data(df):
    education_ranking = {"PhD": 5, "Master's": 4, "Bachelor's": 3, "High School": 2, "Other": 1}
    df['education_rank'] = df['education_level'].map(education_ranking)
    df['subgrade_num'] = df['grade_subgrade'].str[1:].astype(int)
    grade_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
    df['grade_rank'] = df['grade_subgrade'].str[0].map(grade_map)
    df = pd.get_dummies(df, columns=['gender', 'marital_status', 'employment_status', 'loan_purpose'], drop_first=True)
    cols_to_drop = ['id', 'education_level', 'grade_subgrade']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    return df

X_test = prepare_submission_data(test)
expected_features = model.feature_names
X_test = X_test.reindex(columns=expected_features, fill_value=0)

dtest = xgb.DMatrix(X_test)
test_preds = model.predict(dtest)

submission = pd.DataFrame({'id': test_ids, 'loan_paid_back': test_preds})
submission.to_csv('submission_day1.csv', index=False)

C:\Users\VMUser\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


XGBoostError: [17:38:07] C:\actions-runner\_work\xgboost\xgboost\src\common\io.cc:144: Opening loan_repayment_model_day1.json failed: The system cannot find the file specified.

In [19]:
import pandas as pd

def prepare_submission_data2(df):
    df_ex = df.copy()
    df_ex['grade_letter'] = df['grade_subgrade'].str[0]
    df_ex['subgrade_num'] = df['grade_subgrade'].str[1:].astype(int)
    del df_ex['id']
    del df_ex['grade_subgrade']
    grade_map = {'A':1, 'B':2, 'C':3, 'D':4, 'E':5, 'F':6}
    df_ex['grade_letter_rank'] = df_ex['grade_letter'].map(grade_map)
    del df_ex['grade_letter']
    
    education_ranking = {"PhD": 5, "Master's": 4, "Bachelor's": 3, "High School": 2, "Other": 1}
    df_ex['education_rank'] = df_ex['education_level'].map(education_ranking)
    del df_ex['education_level']
    
    df_ex = pd.get_dummies(df_ex, columns=['gender', 'marital_status', 'employment_status', 'loan_purpose'], drop_first=True)

    return df

In [20]:
import joblib
import pandas as pd

df = pd.read_csv('../data/test.csv')
test_ids = df['id']

model = joblib.load('../models/logit_model.pkl')
scaler = joblib.load('../models/scaler.pkl')
expected_features = joblib.load('../models/features.pkl')

X_test = prepare_submission_data2(df)
X_test = X_test.reindex(columns=expected_features, fill_value=0)

X_test = scaler.transform(X_test)

y_pred = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({'id': test_ids, 'loan_paid_back': y_pred})
submission.to_csv('../submissions/logit_submission.csv', index=False)